# Support Vector Machines — Practice Set 3
## eBay iPhones: C, gamma, LinearSVC vs SVC, vs Logistic Regression

### Rules: alpha=0.05, random_state=617, always scale before SVM, do NOT round

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

data = pd.read_csv('eBayClean.csv')
y = data.sold
X = data[['startprice','charCountDescription','upperCaseDescription']]

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X, y, train_size=0.7, random_state=617)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

print('Train:', X_tr.shape, 'Test:', X_te.shape)
print('Sold pct in train:', y_tr.mean()*100)

---
## Question 1
> Fit three LinearSVC models with C=0.01, C=1, C=100.
> What is the train and test accuracy for each?
> Which C gives the best test accuracy?
> (Higher C = narrower margin = more overfitting risk)

In [ ]:
rows = []
for c_val in [0.01, 1, 100]:
    svm = LinearSVC(C=c_val, random_state=617, max_iter=5000)
    svm.fit(X_tr, y_tr)
    rows.append({'C': c_val, 'train_acc': svm.score(X_tr, y_tr), 'test_acc': svm.score(X_te, y_te)})
df = pd.DataFrame(rows)
print(df)
print('Best test acc at C =', df.loc[df.test_acc.idxmax(), 'C'])

---
## Question 2
> Fit three SVC(kernel='rbf') models with gamma=0.001, 0.1, 10 (keep C=1).
> What happens to train accuracy as gamma increases?
> What is the test accuracy for each? Which gamma is best?

In [ ]:
rows = []
for g in [0.001, 0.1, 10]:
    svm = SVC(kernel='rbf', C=1, gamma=g, random_state=617)
    svm.fit(X_tr, y_tr)
    rows.append({'gamma': g, 'train_acc': svm.score(X_tr, y_tr), 'test_acc': svm.score(X_te, y_te)})
df_g = pd.DataFrame(rows)
print(df_g)
# High gamma -> overfitting (train_acc near 1.0, test_acc drops)

---
## Question 3
> Use GridSearchCV with 10-fold CV to jointly tune C and gamma for RBF SVM.
> C = [0.01, 0.1, 1, 10, 100], gamma = [0.001, 0.01, 0.1, 1, 10]
> What is the best C and gamma combination?
> What is the best 10-fold CV score?

In [ ]:
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1, 10]}
grid = GridSearchCV(SVC(kernel='rbf', random_state=617), param_grid, cv=10, n_jobs=-1)
grid.fit(X_tr, y_tr)

print('Best params:   ', grid.best_params_)
print('Best CV score: ', grid.best_score_)
print('Train accuracy:', grid.score(X_tr, y_tr))
print('Test accuracy: ', grid.score(X_te, y_te))

---
## Question 4
> Compare LinearSVC vs SVC(kernel='rbf') speed.
> Which is faster? Why?
> (Hint: LinearSVC is O(m*n); SVC is O(m^2*n) to O(m^3*n))

In [ ]:
import time

t0 = time.time()
LinearSVC(C=1, random_state=617, max_iter=5000).fit(X_tr, y_tr)
print(f'LinearSVC: {time.time()-t0:.4f}s')

t0 = time.time()
SVC(kernel='rbf', C=1, random_state=617).fit(X_tr, y_tr)
print(f'SVC RBF:   {time.time()-t0:.4f}s')
# LinearSVC is faster; use it for large datasets with linear boundaries

---
## Question 5
> Fit LogisticRegression(max_iter=1000, random_state=617) on the same data.
> What is its test accuracy?
> Does SVM (tuned) or Logistic Regression perform better?

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=617)
lr.fit(X_tr, y_tr)

print('LR test accuracy:          ', lr.score(X_te, y_te))
print('Tuned SVM test accuracy:   ', grid.score(X_te, y_te))

---
## Question 6
> Compute AUC for Logistic Regression using predict_proba().
> Can you compute AUC for SVM without predict_proba()? How?
> What is the test AUC for Logistic Regression?

In [ ]:
# LR has predict_proba
lr_proba = lr.predict_proba(X_te)[:, 1]
print('LR AUC:', roc_auc_score(y_te, lr_proba))

# SVM: use decision_function as proxy for AUC
svm_scores = grid.decision_function(X_te)
print('SVM AUC (decision_function):', roc_auc_score(y_te, svm_scores))

# Key: If you need probabilities, prefer Logistic Regression!

---
## Question 7
> Use 5-fold cross_val_score to compare mean accuracy of:
> LinearSVC(C=1), SVC(kernel='rbf', C=1), LogisticRegression(max_iter=1000)
> Which model has the highest mean CV accuracy?

In [ ]:
models = {
    'LinearSVC':    LinearSVC(C=1, random_state=617, max_iter=5000),
    'SVC RBF':      SVC(kernel='rbf', C=1, random_state=617),
    'Logistic Reg': LogisticRegression(max_iter=1000, random_state=617)
}
for name, m in models.items():
    scores = cross_val_score(m, X_tr, y_tr, cv=5)
    print(f'{name:<18} mean={scores.mean():.4f} std={scores.std():.4f}')

---
## Question 8
> Print the classification report for the TUNED SVM on the test set.
> What is the precision for class 1 (sold)?
> What is the recall (sensitivity) for class 1?

In [ ]:
print(classification_report(y_te, grid.predict(X_te)))

---
## Question 9
> When should you choose SVM over Logistic Regression? Select all that apply:
> A. When you need class probabilities
> B. When classes are nearly linearly separable
> C. When dataset has millions of rows
> D. When the boundary is non-linear (RBF kernel)
> E. When you need coefficient p-values
> F. In high-dimensional spaces (text data)

In [ ]:
# A: FALSE - use Logistic Regression for probabilities
# B: TRUE  - SVM excels at clear class separation
# C: FALSE - SVC is slow for large data; use LinearSVC or LR
# D: TRUE  - RBF handles non-linear boundaries well
# E: FALSE - SVM has no p-values (not statistical)
# F: TRUE  - LinearSVC works well in high-dimensional spaces
print('Correct: B, D, F')

---
## Question 10
> Create a final comparison table of ALL models:
> LinearSVC C=0.01, LinearSVC C=1, LinearSVC C=100, SVC RBF default, SVC RBF tuned, Logistic Reg
> Show train and test accuracy.
> Rank by test accuracy. Is the best training model also the best test model?

In [ ]:
all_models = [
    ('LinearSVC C=0.01',  LinearSVC(C=0.01,  random_state=617, max_iter=5000)),
    ('LinearSVC C=1',     LinearSVC(C=1,     random_state=617, max_iter=5000)),
    ('LinearSVC C=100',   LinearSVC(C=100,   random_state=617, max_iter=5000)),
    ('SVC RBF default',   SVC(kernel='rbf',  C=1,  random_state=617)),
    ('Logistic Reg',      LogisticRegression(max_iter=1000, random_state=617)),
]

rows = []
for name, m in all_models:
    m.fit(X_tr, y_tr)
    rows.append({'model': name, 'train_acc': m.score(X_tr, y_tr), 'test_acc': m.score(X_te, y_te)})

rows.append({'model': 'SVC RBF tuned', 'train_acc': grid.score(X_tr, y_tr), 'test_acc': grid.score(X_te, y_te)})

df_final = pd.DataFrame(rows).set_index('model').sort_values('test_acc', ascending=False)
print(df_final)
print('\nBest test model:', df_final.index[0])

---
# Key Takeaways

| SVM vs Logistic Regression | SVM | Logistic Reg |
|---|---|---|
| Class probabilities | No (need probability=True) | Yes (predict_proba) |
| Interpretability | Low (no p-values) | High (coef + p-values) |
| Non-linear boundary | Yes (kernel trick) | Need manual features |
| Speed (large data) | Slow (SVC) | Fast |
| Regularization param | C | C (same meaning) |
| Must scale | Yes (always) | Recommended |

## SVM Exam Rules
1. Higher C = narrower margin = MORE overfitting risk
2. Higher gamma = MORE complex boundary = overfitting
3. ALWAYS fit scaler on train only, transform test
4. LinearSVC faster than SVC for large datasets
5. No predict_proba() for SVM by default
6. GridSearchCV uses cross-validation, NOT the test set